# US Data

References : [Chapter 5 : PCE (BEA)](https://www.bea.gov/resources/methodologies/nipa-handbook/pdf/chapter-05.pdf)

Careful to NIPA levels ; the same tables 2.4.3, 2.4.4 and 2.4.5 without the "U" that stands for "Underlying Details" are different aggregation levels. Those one are the most frequently used, while Lansing and Shapiro use the "Underlying Details" tables (129 components).

## Definition
Personal Consumption Expenditures (PCE) is a measure of the spending on goods and services by people of the United States, constructed and reported by the Bureau of Economic Analysis (BEA). According to the BEA, PCE accounts for about two-thirds of domestic spending and is a significant driver of gross domestic product (GDP).



Its main aggregates are :


We collect :
- **Table 2.4.3U :** Real Personal Consumption Expenditures by Type of Product, Quantity Indexes ; Chain-type quantity index
- **Table 2.4.4U :** Price Indexes for Personal Consumption Expenditures by Type of Product
- **Table 2.4.5U :** Personal Consumption Expenditures by Type of Product

chained value (2017=100) of PCE components indices, real and nominal, as well as quantities at a monthly frequency. Data is seasonally adjusted. 


### Index sub-components


In [1]:
# Librairies
import json
import pandas as pd
from fismi import sdxData as sdx
import numpy as np

In [2]:
with open('/Users/lea_gosselin/.openbb_platform/user_settings.json', 'r') as file:
    keys = json.load(file)

In [3]:
url = "https://apps.bea.gov/api/data"
userid = keys["credentials"].get("bea_api_key")

# Load PCE data from BEA

# Query data
index_raw = sdx.getBeaData("U20403", userid)   # Table 2.4.3U : Real Personal Consumption Expenditures by Type of Product, Quantity Indexes
price_raw   = sdx.getBeaData("U20404", userid)   # Table 2.4.4U : Price Indexes for Personal Consumption Expenditures by Type of Product
weights_raw = sdx.getBeaData("U20405", userid)   # Table 2.4.5U : Personal Consumption Expenditures by Type of Product


In [4]:
# Weights PCE (share of the expenditure in the total, as current $) 
df_price   = price_raw.drop_duplicates(subset=["TIME_PERIOD","SeriesCode"]).pivot(index="TIME_PERIOD", columns="LineDescription", values="DataValue")
df_weights = weights_raw.drop_duplicates(subset=["TIME_PERIOD","SeriesCode"]).pivot(index="TIME_PERIOD", columns="SeriesCode", values="DataValue")

In [5]:
# Load PCE components levels (not available online)
levelPce = pd.read_excel("PCEComponentsLevel.xlsx", dtype={"LineNumber": str, "Level": str})
levelPce[levelPce["Level"]=="4"]

,LineNumber,Level,LineDescription,SeriesCode
5,6,4,New autos,DNEARA
8,9,4,New light trucks,DNWTRA
12,13,4,Used autos,DNPURA
16,17,4,Used light trucks,DUTRRA
20,21,4,Tires,DTATRA
...,...,...,...,...
345,346,4,"Nonprofit hospitals, gross output",DHSORA
346,347,4,"Nonprofit nursing homes, gross output",DNXORA
357,358,4,Outpatient services to households,DOUSRA
358,359,4,Nonprofit hospitals services to househ...,DNPHRA


In [ ]:
# Merge data on LineNumber
index_raw.loc[index_raw["LineNumber"].isin(levelPce.loc[levelPce["Level"]=="4", "LineNumber"]), ["LineNumber","LineDescription","SeriesCode"]].drop_duplicates()
index_raw = index_raw.merge(
    levelPce[["LineNumber","Level"]],
    how="left",
    on="LineNumber")
index_raw

,TIME_PERIOD,LineNumber,LineDescription,SeriesCode,DataValue,Level_x,Level_y,Level
0,1959-01-01,1,Personal consumption expenditures,DPCERA,15.188,0,0,0
1,1959-02-01,1,Personal consumption expenditures,DPCERA,15.346,0,0,0
2,1959-03-01,1,Personal consumption expenditures,DPCERA,15.491,0,0,0
3,1959-04-01,1,Personal consumption expenditures,DPCERA,15.435,0,0,0
4,1959-05-01,1,Personal consumption expenditures,DPCERA,15.622,0,0,0
...,...,...,...,...,...,...,...,...
806,2026-03-01,1,Personal consumption expenditures,DPCERA,125.879,0,0,0
807,2026-04-01,1,Personal consumption expenditures,DPCERA,126.130,0,0,0
808,2026-05-01,1,Personal consumption expenditures,DPCERA,126.597,0,0,0
809,2026-06-01,1,Personal consumption expenditures,DPCERA,127.157,0,0,0
